# TinyDeploy walkthrough

Companion notebook to *TinyML for Resource-Constrained IoT Devices: Techniques, Benchmarks, and a Unified Deployment Framework* (J. ET-TOUSY, 2026).

It runs the TinyDeploy pipeline end to end and shows, for each stage, the inputs, the decisions, the logs and the outputs:

| Part | Paper | What is shown |
|---|---|---|
| 1 | Fig. 1, Table II | Pipeline stages and responsibility boundaries |
| 2 | Algorithm 1 | 2 x 4 compression grid, support checks, budget filter, selection, Pareto set, execution logs, C header |
| 3 | Algorithm 1 | Three-candidate example: rejection reasons, Pareto set, regression check |
| 4 | OTA scheduler, Table VI | Delta codec, A/B update with rollback, 500-device fleet simulation |
| 5 | - | Host-side overhead of TinyDeploy itself (selection, CLI call, OTA simulation, memory) |
| 6 | Section V-B | Importing Edge Impulse reports and on-board measurements as profiles |
| 7 | Section III | Optional TensorFlow adapter: INT8 PTQ, structured pruning, distillation |

**Runtime:** under a minute on a laptop CPU (Part 7 needs TensorFlow and is skipped without it).
**Data:** Parts 2-4 use small generated example inputs, labelled `synthetic` in their provenance, to show how each stage behaves. They do not reproduce the paper's results, which come from trained models run on the STM32L476 and Nano 33 BLE Sense boards with Edge Impulse.

**Setup:** from the repository root, `pip install -e ".[notebook]"` (add `tensorflow` for Part 7).

In [ ]:
import json
import platform
import shutil
import statistics
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import tinydeploy
from tinydeploy import apply_delta, make_delta, run, simulate
from tinydeploy._util import digest, read_json, write_json
from tinydeploy.delta import wire
from tinydeploy.demo import EXPECTED_EXAMPLE, prepare_grid, prepare_ota, prepare_selection
from tinydeploy.ota import Device

OUT = Path("outputs")
shutil.rmtree(OUT, ignore_errors=True)  # TinyDeploy never overwrites a run, so start clean
OUT.mkdir()

pd.set_option("display.width", 160)
print(f"TinyDeploy {tinydeploy.__version__} | Python {platform.python_version()} | {platform.platform()}")
print(f"CPU: {platform.processor() or platform.machine()}")

## 1. Pipeline and responsibility boundaries

| Stage | Module | External tools (not TinyDeploy) | TinyDeploy |
|---|---|---|---|
| Candidate generation | `grid`, `compression` | Edge Impulse / EON Tuner, TFLite converter | Grid expansion, support checks, logged compressor calls |
| Profiling | `profiles` | Edge Impulse build report and on-device runner, on-board timing | Profile schema, provenance, SHA-256 binding |
| Filtering and selection | `selection` | - | Budget filter, energy `E = t x I x V`, Pareto set, selection |
| Target build | `build` | TFLite Micro, CMSIS-NN, board toolchain | C array, shell-free build call, build log, firmware hash |
| OTA preparation | `delta`, `ota` | Radio stack, bootloader | Delta codec, A/B update logic, fleet simulation |

## 2. Algorithm 1: constraint-based candidate selection

The grid covers bit-width {8, 4} x pruning ratio {0, 0.3, 0.5, 0.7}. The target below resembles the STM32L476 (128 kB SRAM, 1 MB flash, 80 MHz Cortex-M4F) and, like TFLM/CMSIS-NN on Cortex-M4, runs INT8 kernels only, so the four INT4 configurations are skipped with a recorded reason.

Compression and profiling are external commands. Here a small stand-in adapter (`tinydeploy._demo_adapter`) replaces Edge Impulse so the notebook runs anywhere; in a real deployment the same two command slots call the compressor and the profiler.

In [ ]:
grid_config = prepare_grid(OUT / "inputs" / "grid")
config = read_json(grid_config)
print(json.dumps({k: config[k] for k in ("metric", "minimum_score", "target")}, indent=1))
print("compress:", " ".join(config["grid"]["compress"][1:]))
print("profile: ", " ".join(config["grid"]["profile"][1:]))

In [ ]:
grid = run(grid_config, OUT / "grid", allow_synthetic=True)

evaluated = pd.DataFrame([{
    "candidate": c["id"], "bits": c["bits"], "prune": c["prune_ratio"],
    "size_kB": c["model_size_bytes"] / 1024, "sram_kB": c["peak_sram_bytes"] / 1024,
    "latency_ms": c["latency_ms"], "energy_uJ": c["estimated_energy_uj"], "score": c["score"],
    "rejected_for": ", ".join(c["rejection_reasons"]) or "-",
} for c in grid["candidates"]])
skipped = pd.DataFrame(grid["skipped"]).rename(columns={"id": "candidate"})

print(f"evaluated: {len(evaluated)}   skipped: {len(skipped)}   feasible: {grid['feasible_ids']}")
print(f"selected (max score among feasible): {grid['selected']}   Pareto set: {grid['pareto_ids']}")
display(evaluated.round(2))
display(skipped)

In [ ]:
target = config["target"]
color = {cid: ("tab:green" if cid == grid["selected"] else
               "tab:blue" if cid in grid["feasible_ids"] else "tab:red") for cid in evaluated.candidate}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.scatter(evaluated.sram_kB, evaluated.latency_ms, c=[color[c] for c in evaluated.candidate], s=70)
ax1.axvline(target["sram_bytes"] / 1024, ls="--", c="gray")
ax1.axhline(target["max_latency_ms"], ls="--", c="gray")
ax1.set(xlabel="peak SRAM (kB)", ylabel="latency (ms)", title="Hardware budgets (dashed)")
ax2.scatter(evaluated.size_kB, evaluated.score, c=[color[c] for c in evaluated.candidate], s=70)
ax2.axhline(config["minimum_score"], ls="--", c="gray")
ax2.set(xlabel="model size (kB)", ylabel="validation score", title="Score vs size (dashed: minimum score)")
for ax, (x, y) in ((ax1, ("sram_kB", "latency_ms")), (ax2, ("size_kB", "score"))):
    for _, row in evaluated.iterrows():
        ax.annotate(row.candidate, (row[x], row[y]), textcoords="offset points", xytext=(5, 4), fontsize=8)
fig.legend(handles=[plt.Line2D([], [], marker="o", ls="", c=c, label=l) for c, l in
                    (("tab:green", "selected"), ("tab:blue", "feasible"), ("tab:red", "rejected"))],
           loc="lower center", ncol=3)
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

### Execution logs and outputs

Every external command is run without a shell and logged with its argument vector, return code, elapsed time, stdout and stderr. Profiles are archived and bound to their model by SHA-256, and the selected model is emitted as a C array for the board's TFLM project.

In [ ]:
run_dir = OUT / "grid"
print(sorted(p.relative_to(run_dir).as_posix() for p in run_dir.rglob("*") if p.is_file())[:14], "...")
log = read_json(run_dir / "logs" / f"{grid['selected']}-compress.json")
print(json.dumps({k: log[k] for k in ("argv", "returncode", "elapsed_s", "stdout")}, indent=1))
print("".join((run_dir / "model_data.h").read_text().splitlines(keepends=True)[:8]))

## 3. Three-candidate example

Three candidates against a small target: two fit, `over_budget` violates the SRAM and latency budgets. The example is seeded, so the checks below compare this run with its expected outcome.

In [ ]:
selection = run(prepare_selection(OUT / "inputs" / "selection"), OUT / "selection", allow_synthetic=True)
display(pd.DataFrame([{
    "candidate": c["id"], "size_B": c["model_size_bytes"], "sram_B": c["peak_sram_bytes"],
    "latency_ms": c["latency_ms"], "energy_uJ": c["estimated_energy_uj"], "score": c["score"],
    "rejected_for": ", ".join(c["rejection_reasons"]) or "-"} for c in selection["candidates"]]))

expected = EXPECTED_EXAMPLE["selection"]
rows = {c["id"]: c for c in selection["candidates"]}
checks = {
    "candidates": (expected["candidates"], len(selection["candidates"])),
    "feasible": (expected["feasible"], len(selection["feasible_ids"])),
    "Pareto": (expected["pareto"], len(selection["pareto_ids"])),
    "selected": (expected["selected"], selection["selected"]),
    "over_budget rejected for": (expected["over_budget_rejections"], rows["over_budget"]["rejection_reasons"]),
}
display(pd.DataFrame([{"quantity": k, "expected": str(p), "this run": str(r), "match": p == r}
                      for k, (p, r) in checks.items()]))
assert all(p == r for p, r in checks.values())

## 4. OTA update preparation

### 4.1 Hash-bound delta codec
A delta lists only the changed byte runs and carries the SHA-256 of both images, so it cannot be applied to the wrong base or produce an unexpected image.

In [ ]:
old_path, new_path, ota_config_path = prepare_ota(OUT / "inputs" / "ota")
old, new = old_path.read_bytes(), new_path.read_bytes()
delta = make_delta(old, new)
print(f"full image: {len(new)} B   delta on the wire: {len(wire(delta))} B   runs: {len(delta['runs'])}")
assert apply_delta(old, delta) == new
print("round trip: OK")

for name, attempt in [("wrong base image", lambda: apply_delta(new, delta)),
                      ("tampered payload", lambda: apply_delta(old, {**delta, "runs": [[1024, "AAAA"]]}))]:
    try:
        attempt()
    except tinydeploy.TinyDeployError as exc:
        print(f"{name}: rejected ({exc})")

### 4.2 A/B update with self-test and rollback
The image is written to the inactive slot, verified, and committed only if the post-boot self-test passes; otherwise the device keeps its previous image.

In [ ]:
device = Device(old, flash_bytes=1048576, bootloader_bytes=32768, metadata_bytes=4096, scratch_bytes=16384)
print(f"slot size: {device.slot_bytes} B")
ok = lambda image: digest(image) == digest(new)
print("valid update        ->", device.update(delta, digest(new), ok, delta=True))
print("corrupted after verify ->", device.update(make_delta(new, old), digest(old),
                                                 lambda image: digest(image) == digest(old),
                                                 delta=True, corrupt_after_verify=True))
print("active image is still the new one:", device.image == new)
print("wrong hash          ->", device.update(old, "0" * 64, ok))

### 4.3 Fleet simulation
500 devices, 3 update rounds, 9.6 kbps link, 2% packet loss, up to 3 retries, 0.4% post-verification fault probability. Full-image and delta modes use identical random streams.

In [ ]:
ota_config = read_json(ota_config_path)
start = time.perf_counter()
ota = simulate(old_path, new_path, ota_config, OUT / "ota")
ota_seconds = time.perf_counter() - start

modes = pd.DataFrame(ota["modes"]).T
modes["on_air_MB"] = modes.transmitted_bytes / 1e6
display(modes[["events", "committed", "rolled_back", "transfer_failed", "on_air_MB",
               "radio_time_s", "estimated_radio_energy_mj", "rollback_rate"]].round(4))
print(f"delta radio-energy saving: {ota['delta_energy_saving_fraction']:.1%}   (simulated in {ota_seconds:.2f} s)")

expected = EXPECTED_EXAMPLE["ota"]
checks = {
    "events per mode": (expected["events_per_mode"], int(modes.events["full"])),
    "commits per mode": (expected["committed"], int(modes.committed["delta"])),
    "rollbacks per mode": (expected["rolled_back"], int(modes.rolled_back["delta"])),
    "full image on air (MB)": (expected["full_mb_on_air"], round(modes.on_air_MB["full"], 2)),
    "delta on air (MB)": (expected["delta_mb_on_air"], round(modes.on_air_MB["delta"], 2)),
    "energy saving (%)": (expected["energy_saving_pct"], round(100 * ota["delta_energy_saving_fraction"], 1)),
}
display(pd.DataFrame([{"quantity": k, "expected": p, "this run": r, "match": p == r} for k, (p, r) in checks.items()]))
assert all(p == r for p, r in checks.values())

In [ ]:
events = pd.read_json(OUT / "ota" / "events.jsonl", lines=True)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
modes.on_air_MB.plot.bar(ax=ax1, color=["tab:gray", "tab:green"], rot=0, title="Bytes on air (MB)")
(modes.estimated_radio_energy_mj / 1000).plot.bar(ax=ax2, color=["tab:gray", "tab:green"], rot=0,
                                                   title="Radio energy (J)")
fig.tight_layout()
plt.show()

print("status counts per mode:")
display(events.groupby(["mode", "status"]).size().unstack(fill_value=0))
print("rolled-back events (injected post-verification faults):")
display(events[events.status == "rolled_back"].head(10))

## 5. Host-side overhead

Cost of TinyDeploy's own orchestration only; model training, profiling and firmware compilation are excluded. Absolute times depend on the machine.

In [ ]:
def timed_runs(n=10):
    selection_ms, cli_ms = [], []
    cfg = prepare_selection(OUT / "inputs" / "timing")
    for i in range(n):
        t = time.perf_counter()
        run(cfg, OUT / "timing" / f"api-{i}", allow_synthetic=True)
        selection_ms.append(1000 * (time.perf_counter() - t))
        t = time.perf_counter()
        subprocess.run([sys.executable, "-m", "tinydeploy", "run", str(cfg), "--output",
                        str(OUT / "timing" / f"cli-{i}"), "--allow-synthetic"], check=True,
                       capture_output=True)
        cli_ms.append(1000 * (time.perf_counter() - t))
    return selection_ms, cli_ms

selection_ms, cli_ms = timed_runs()
rows = [("selection pipeline, in-process (ms)", selection_ms),
        ("CLI call incl. interpreter start-up (ms)", cli_ms)]
overhead = pd.DataFrame([{"measurement": name, "mean": statistics.mean(v), "stdev": statistics.stdev(v), "runs": len(v)}
                         for name, v in rows])
overhead.loc[len(overhead)] = ["OTA simulation, 3,000 events (s)", ota_seconds, float("nan"), 1]
display(overhead.round(3))

try:
    import resource  # Linux and macOS
    cmd = [sys.executable, "-m", "tinydeploy", "simulate-ota", str(old_path), str(new_path),
           "--config", str(ota_config_path), "--output", str(OUT / "ota-memory")]
    subprocess.run(cmd, check=True, capture_output=True)
    peak = resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss
    peak_mb = peak / (1024 * 1024 if sys.platform == "darwin" else 1024)  # bytes on macOS, kB on Linux
    print(f"peak resident memory of the OTA simulation process: {peak_mb:.0f} MB")
except ImportError:
    print("peak-memory measurement uses the 'resource' module (not available on Windows)")

## 6. Importing real profiles

In the paper, latency was measured on the boards with the Edge Impulse runner, and RAM and flash come from the Edge Impulse build report. TinyDeploy turns an Edge Impulse profiling report, or on-board timing samples from `firmware/tinydeploy_benchmark.h`, into a hash-bound profile with explicit provenance. The report below has the structure of an Edge Impulse INT8 profiling response for one device.

In [ ]:
from tinydeploy.profiles import from_benchmark_csv, from_edge_impulse, validate_profile

ex = OUT / "inputs" / "import"
ex.mkdir(parents=True)
(ex / "model.tflite").write_bytes(b"example model bytes")
write_json(ex / "edge_impulse_report.json", {"success": True, "model": {"profileInfo": {"int8": {
    "device": "st-iot-discovery-kit", "timePerInferenceMs": 90, "isSupportedOnMcu": True,
    "memory": {"eon": {"ram": 61440, "rom": 84992}, "tflite": {"ram": 70656, "rom": 102400}}}}}})
ei_profile = from_edge_impulse(ex / "edge_impulse_report.json", ex / "model.tflite",
                               target="stm32l476", metric="accuracy", score=0.91)
print(json.dumps({k: ei_profile[k] for k in ("latency_ms", "peak_sram_bytes", "flash_bytes", "provenance")}, indent=1))

# On-board protocol: 30 timed repetitions after warm-up, DWT cycle counter at 80 MHz.
rows = ["sample_index,elapsed_ticks,timer_hz"] + [f"{i},{7_200_000 + 4_000 * (i % 5)},80000000" for i in range(30)]
(ex / "timing.csv").write_text("\n".join(rows) + "\n")
protocol = {"warmup": 5, "repetitions": 30, "timer": "DWT CYCCNT", "compiler": "arm-none-eabi-gcc 13.2",
            "flags": "-O3 -mcpu=cortex-m4", "clock_hz": 80_000_000}
measured = validate_profile(from_benchmark_csv(ex / "timing.csv", ex / "model.tflite", target="stm32l476",
                                               metric="accuracy", score=0.91, peak_sram_bytes=61440,
                                               protocol=protocol), ex)
print(f"measured latency: {measured['latency_ms']:.3f} ms +/- {measured['latency_stdev_ms']:.3f} ms "
      f"(min {measured['latency_min_ms']:.3f}, max {measured['latency_max_ms']:.3f}, n={len(measured['latency_samples_ms'])})")

## 7. Optional: TensorFlow compression adapter

INT8 full-integer post-training quantization, structured pruning of Dense-only models and teacher-student distillation, on a small synthetic task. Skipped when TensorFlow is not installed.

In [ ]:
try:
    import numpy as np
    import tensorflow as tf
except ImportError:
    tf = None
    print("TensorFlow not installed - skipping Part 7 (pip install -e '.[tensorflow]').")

if tf is not None:
    from tinydeploy.compression import distill, int8_ptq, prune_dense

    rng = np.random.default_rng(0)
    x = rng.normal(size=(2048, 16)).astype("float32")
    y = (x[:, 0] > 0).astype("int64") + 2 * (x[:, 1] > 0).astype("int64")
    x_train, y_train, x_val, y_val = x[:1536], y[:1536], x[1536:], y[1536:]

    def mlp(units):
        return tf.keras.Sequential([tf.keras.Input(shape=(16,)),
                                    tf.keras.layers.Dense(units, activation="relu"),
                                    tf.keras.layers.Dense(units, activation="relu"),
                                    tf.keras.layers.Dense(4, activation="softmax")])

    def int8_accuracy(blob):
        interp = tf.lite.Interpreter(model_content=blob)
        interp.allocate_tensors()
        inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
        scale, zero = inp["quantization"]
        correct = 0
        for sample, label in zip(x_val, y_val):
            q = np.clip(np.round(sample / scale + zero), -128, 127).astype(np.int8)[None]
            interp.set_tensor(inp["index"], q)
            interp.invoke()
            correct += int(np.argmax(interp.get_tensor(out["index"])) == label)
        return correct / len(y_val)

    tf.keras.utils.set_random_seed(0)
    teacher = mlp(128)
    teacher.compile("adam", "sparse_categorical_crossentropy")
    teacher.fit(x_train, y_train, epochs=10, verbose=0)

    results = []
    for name, model in [("teacher FP32 -> INT8", teacher),
                        ("pruned 50% -> INT8", prune_dense(teacher, 0.5))]:
        blob = int8_ptq(model, x_train[:256], OUT / f"{name.split()[0]}.tflite")
        results.append({"model": name, "params": model.count_params(), "tflite_B": len(blob),
                        "int8_val_acc": int8_accuracy(blob)})
    student = mlp(16)
    history = distill(teacher, student, x_train, y_train, epochs=10, temperature=3.0, alpha=0.5)
    blob = int8_ptq(student, x_train[:256], OUT / "student.tflite")
    results.append({"model": "KD student -> INT8", "params": student.count_params(), "tflite_B": len(blob),
                    "int8_val_acc": int8_accuracy(blob)})
    display(pd.DataFrame(results))

## Scope

- TinyDeploy runs on the host. The OTA part simulates A/B bootloader logic; it is not bootloader firmware, and its energy model counts radio airtime only (assumptions are listed in each `summary.json`).
- Compression, profiling and kernels are delegated to external tools; the built-in TensorFlow pruning supports Dense-only `Sequential` models.
- The grid is finite, so the Pareto set is reported alongside the selection rather than claiming global optimality.

All outputs of this notebook are in `outputs/`.